# NLP #2: Text Preprocessing

**AI Teacher** · Interview prep notebook for the video *Text Preprocessing in NLP* (taught by Gonu Jha Ji with Priya, Rahul and Sneha).

Run the cells from top to bottom (`Shift + Enter`). Every output you see was produced by running this notebook. Change the examples and run them again - that is the best way to learn.

## What you will learn
1. Why we preprocess text (garbage in, garbage out)
2. Each step with code: lowercasing, regex cleaning, stopwords, contractions, emojis, slang
3. **The hidden bug**: removing "not" flips the meaning of a review
4. A reusable `clean_text()` function - heavy cleaning for classical ML, light cleaning for transformers
5. The interview questions from the video, with model answers

In [1]:
# One-time setup (uncomment if a library is missing):
# !pip install nltk scikit-learn pandas
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
print("Ready!")

Ready!


## 1. Key definitions

| Term | Meaning |
|---|---|
| **Text preprocessing** | Cleaning and normalizing raw text before feature extraction |
| **Normalization** | Making different forms of the same thing identical: `Apple`, `APPLE` → `apple` |
| **Regex** (regular expression) | A pattern language for finding / replacing text: `https?://\S+` matches URLs |
| **Stopwords** | Very common words with little meaning on their own: *the, is, a, of* |
| **Vocabulary** | The set of unique tokens in a corpus. Cleaner text → smaller vocabulary → faster, better models |

> **Gonu Jha's story:** the messenger dropped the "small words" and *"Do NOT buy salt today"* became *"Buy salt today"*. Small words can carry BIG meaning.

## 2. Lowercasing
`Apple`, `apple` and `APPLE` become one token instead of three - a smaller vocabulary.

In [2]:
words = ["Apple", "apple", "APPLE", "Good", "good"]
print("before:", len(set(words)), "unique ->", set(words))
print("after :", len({w.lower() for w in words}), "unique ->", {w.lower() for w in words})

before: 5 unique -> {'Good', 'good', 'APPLE', 'Apple', 'apple'}
after : 2 unique -> {'apple', 'good'}


**When NOT to lowercase:** Named Entity Recognition (*Apple* the company vs *apple* the fruit), acronyms
(*US* vs *us*, *IT* vs *it*), cased transformer models (`bert-base-cased`), and shouting as emotion
("I am VERY angry").

## 3. Regex cleaning
`re.sub(pattern, replacement, text)` replaces every match of `pattern`.

| Remove | Pattern |
|---|---|
| HTML tags | `<.*?>` |
| URLs | `https?://\S+` |
| @mentions | `@\w+` |
| Hashtag symbol only (keep the word) | `#(\w+)` → `\1` |
| Everything except letters and spaces | `[^a-z\s]` |

In [3]:
import re
raw = "<p>Check out https://abc.com NOW!!! @zomato #IPL The food was NOT good :(</p>"
step = raw.lower();                          print("lower   :", step)
step = re.sub(r"<.*?>", " ", step);          print("no html :", step)
step = re.sub(r"https?://\S+", " ", step);   print("no urls :", step)
step = re.sub(r"@\w+", " ", step);           print("no @    :", step)
step = re.sub(r"#(\w+)", r"\1", step);       print("hashtag :", step)
step = re.sub(r"[^a-z\s]", " ", step);       print("letters :", step)
step = re.sub(r"\s+", " ", step).strip();    print("spaces  :", step)

lower   : <p>check out https://abc.com now!!! @zomato #ipl the food was not good :(</p>
no html :  check out https://abc.com now!!! @zomato #ipl the food was not good :( 
no urls :  check out   now!!! @zomato #ipl the food was not good :( 
no @    :  check out   now!!!   #ipl the food was not good :( 
hashtag :  check out   now!!!   ipl the food was not good :( 
letters :  check out   now      ipl the food was not good    
spaces  : check out now ipl the food was not good


## 4. Stopwords - and the hidden bug
NLTK's English stopword list has 198 words. Look carefully: it contains **"not"**!

In [4]:
from nltk.corpus import stopwords
stop = set(stopwords.words("english"))
print(len(stop), "stopwords")
print("negations inside the list:", sorted(w for w in stop if w in {"not", "no", "nor", "don't", "isn't", "wasn't"}))

198 stopwords
negations inside the list: ["don't", "isn't", 'no', 'nor', 'not', "wasn't"]


In [5]:
# The full cleaning pipeline from the video
text = "<p>Check out https://abc.com NOW!!! The food was NOT good :(</p>"
text = text.lower()
text = re.sub(r"<.*?>", " ", text)          # HTML tags
text = re.sub(r"https?://\S+", " ", text)   # URLs
text = re.sub(r"[^a-z\s]", " ", text)       # punctuation, numbers
tokens = text.split()
print("tokens          :", tokens)
print("stopwords gone  :", [w for w in tokens if w not in stop])

tokens          : ['check', 'out', 'now', 'the', 'food', 'was', 'not', 'good']
stopwords gone  : ['check', 'food', 'good']


`food good` - a **negative** review now looks **positive**! The fix: keep the negations.

In [6]:
stop_keep_neg = stop - {"not", "no", "nor"}
print([w for w in tokens if w not in stop_keep_neg])

['check', 'food', 'not', 'good']


**When to remove stopwords:** search, topic modeling, keyword extraction, BoW / TF-IDF features.
**When to keep them:** sentiment (*not good*), question answering, translation, and any transformer / LLM.

## 5. Other useful steps
### 5a. Expand contractions
`don't` → `do not` makes the negation visible as a separate word.

In [7]:
CONTRACTIONS = {"don't": "do not", "doesn't": "does not", "isn't": "is not", "wasn't": "was not",
                "can't": "cannot", "won't": "will not", "i'm": "i am", "it's": "it is"}

def expand(text):
    pattern = re.compile(r"\b(" + "|".join(map(re.escape, CONTRACTIONS)) + r")\b")
    return pattern.sub(lambda m: CONTRACTIONS[m.group(0)], text.lower())

print(expand("The pizza wasn't hot and I'm not happy, it's cold"))

the pizza was not hot and i am not happy, it is cold


### 5b. Emojis and emoticons
For sentiment, an emoji is a strong signal - convert it to a word instead of deleting it.

In [8]:
EMOJI = {":)": " happy ", ":(": " sad ", "😍": " love ", "😡": " angry ", "👍": " good "}

def emoji_to_text(text):
    for e, word in EMOJI.items():
        text = text.replace(e, word)
    return re.sub(r"\s+", " ", text).strip()

print(emoji_to_text("Biryani 😍 but delivery 😡 :("))

Biryani love but delivery angry sad


### 5c. Elongated words and slang (very common in Indian social media)

In [9]:
SLANG = {"gr8": "great", "gud": "good", "pls": "please", "thx": "thanks", "acha": "accha"}

def normalize(text):
    text = re.sub(r"(\w)\1{2,}", r"\1\1", text.lower())   # sooo -> soo, goood -> good
    return " ".join(SLANG.get(w, w) for w in text.split())

print(normalize("sooo goood, gr8 taste thx"))

soo good, great taste thanks


## 6. One reusable function: classical ML vs transformers
- **TF-IDF + Naive Bayes / SVM:** heavy cleaning helps.
- **BERT / GPT:** minimal cleaning - their tokenizer handles case, punctuation and rare words. Removing stopwords or punctuation can *hurt* them.
- **Always:** remove HTML, boilerplate, duplicates and personal data (phone numbers, emails).

In [10]:
def clean_text(text, mode="classical"):
    text = re.sub(r"<.*?>", " ", text)                    # always: HTML
    text = re.sub(r"https?://\S+|@\w+", " ", text)       # always: URLs, mentions
    text = re.sub(r"\b[6-9]\d{9}\b", " <PHONE> ", text)   # always: personal data
    if mode == "transformer":
        return re.sub(r"\s+", " ", text).strip()          # keep case, punctuation, stopwords
    text = emoji_to_text(expand(text))
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^a-z\s]", " ", normalize(text))
    return " ".join(w for w in text.split() if w not in stop_keep_neg)

tweets = ["<b>Pizza</b> wasn't gud :( call 9876543210 https://t.co/x",
          "@swiggy sooo fast delivery!!! #Happy 😍"]
for t in tweets:
    print("classical  :", clean_text(t))
    print("transformer:", clean_text(t, "transformer"))
    print()

classical  : pizza not good sad call phone
transformer: Pizza wasn't gud :( call <PHONE>

classical  : soo fast delivery happy love
transformer: sooo fast delivery!!! #Happy 😍



## 7. Interview questions (from the video)

**Q1. What are the common text preprocessing steps?**
Lowercasing; removing HTML, URLs and punctuation; tokenization; stopword removal; stemming or lemmatization.
Then add: *"it depends on the task and the model"* - that sentence shows maturity.

**Q2. Should you always remove stopwords?**
No. Remove for search, topic modeling, keyword extraction. Keep for sentiment (*not good*), QA, translation
and LLMs. Customize the list: keep negations like *not, no, never*.

**Q3. When should you NOT lowercase text?**
NER (*Apple* vs *apple*), acronyms (*US* vs *us*), cased transformer models, and when capitals carry
emotion (*VERY angry*).

**Q4. How would you clean tweets or social media text?**
Remove URLs and @mentions; keep hashtag words (#IPL → IPL); normalize elongated words (*sooo goood* → *so good*);
convert emojis and slang instead of deleting them; handle Hinglish / code-mixed text; keep negations.

### Common mistakes
- Blindly removing stopwords and losing "not"
- Heavy cleaning before feeding text to BERT or GPT
- Removing numbers that matter: prices, ratings, dates
- Using different cleaning in training and in production

## 8. Practice
**Task (from the video):** write a regex that removes @mentions but keeps the hashtag words.

In [11]:
# Solution
tweet = "@virat @bcci what a match! #IPL #CSKvsMI"
print(re.sub(r"#(\w+)", r"\1", re.sub(r"@\w+\s*", "", tweet)))

what a match! IPL CSKvsMI


**Next video:** NLP #3 - Tokenization.